# Unity Catalog and volumes

Unity Catalog stores tables in three levels: **catalog**, **schema**, and **table**. A name such as `workspace.data_ingestion.customers` is the catalog, then the schema, then the table. This notebook lists catalogs, creates the `data_ingestion` schema, and checks which catalog and schema are in use.

`DESCRIBE SCHEMA` and `DESCRIBE VOLUME` print the details of one object, such as its owner and location.

A volume is the file side of the data lake. `CREATE VOLUME raw_files` creates a place to store files, such as a CSV, under `/Volumes/workspace/data_ingestion/raw_files/`.

In [0]:
%sql
SHOW CATALOGS;

catalog
samples
system
workspace


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS
  workspace.data_ingestion;

In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA data_ingestion;

In [0]:
%sql
SELECT
    current_catalog() AS catalog_name,
    current_schema() AS schema_name;

catalog_name,schema_name
workspace,data_ingestion


In [0]:
%sql
SHOW SCHEMAS IN workspace;

databaseName
data_ingestion
dataexpert_student
default
information_schema


In [0]:
%sql
DESCRIBE SCHEMA EXTENDED data_ingestion;

database_description_item,database_description_value
Catalog Name,workspace
Namespace Name,data_ingestion
Comment,
Location,
Owner,example@gmail.com
Properties,"((unity.catalog.managed.delta.defaults.delta.parquet.format.version,2.12.0), (unity.catalog.managed.delta.defaults.delta.parquet.format.version.afe.internal,2.12.0), (unity.catalog.managed.iceberg.defaults.delta.feature.catalogManaged,supported))"
Predictive Optimization,ENABLE (inherited from METASTORE metastore_aws_us_east_2)


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS raw_files;

In [0]:
%sql
SHOW VOLUMES;

database,volume_name
data_ingestion,raw_files


In [0]:
%sql
DESCRIBE VOLUME raw_files;

name,catalog,database,owner,storage_location,volume_type,comment,securable_type,securable_kind
raw_files,workspace,data_ingestion,example@gmail.com,,MANAGED,null,VOLUME,VOLUME_DB_STORAGE


In [0]:
%python
file_path = "/Volumes/workspace/data_ingestion/raw_files/customers.csv"

csv_data = """id,name,country
1,Alice,Finland
2,Bob,Sweden
3,Anna,Finland
"""

dbutils.fs.put(file_path, csv_data, overwrite=True)

Wrote 60 bytes.


True

In [0]:
%sql
LIST '/Volumes/workspace/data_ingestion/raw_files/';

path,name,size,modification_time
/Volumes/workspace/data_ingestion/raw_files/customers.csv,customers.csv,60,1791705517000


In [0]:
%fs ls /Volumes/workspace/data_ingestion/raw_files/

path,name,size,modificationTime
dbfs:/Volumes/workspace/data_ingestion/raw_files/customers.csv,customers.csv,60,1791705517000


In [0]:
df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/workspace/data_ingestion/raw_files/customers.csv")
)

display(df)

id,name,country
1,Alice,Finland
2,Bob,Sweden
3,Anna,Finland


In [0]:
%sql
SHOW FUNCTIONS IN workspace.data_ingestion;

function
!
!=
%
&
*
+
-
/
<
<<
